<a href="https://colab.research.google.com/github/WHENKEY2007/AgenticAI/blob/main/AgenticAI-Day2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install openai python-dotenv ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 16.3 MB/s  0:00:00


In [2]:
from openai import OpenAI
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "openai/gpt-oss-120b"

print("HealthTrack AI connected successfully!")

HealthTrack AI connected successfully!


In [3]:
SYSTEM_PROMPT = """
You are HealthTrack AI, a friendly personal health assistant.

ROLE:
Help users organize basic health information
and understand general wellness concepts.

FACTS:
- Encourage healthy sleep habits.
- Encourage regular physical activity.
- Encourage balanced nutrition.
- Remind users to follow professional medical advice.

RULES:
- Never diagnose diseases.
- Never prescribe medicines or dosages.
- Never recommend stopping prescribed medication.
- For serious or emergency symptoms, advise seeking
  immediate professional medical assistance.
- Do not invent medical information.
- Ask for clarification when necessary.

FORMAT:
- Use simple English.
- Give short and understandable answers.
- Use bullet points when helpful.
"""

In [4]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "How can I improve my sleep routine?"}
    ]
)

print(response.choices[0].message.content)

Here are some easy steps to help you build a better sleep routine:

- **Set a regular schedule**
  - Go to bed and wake up at the same time every day, even on weekends.
- **Create a calming bedtime routine**
  - Dim the lights, read a book, or do gentle stretching for 15‑30 minutes before sleep.
- **Limit screens**
  - Turn off phones, tablets, and computers at least 30 minutes before bed; the blue light can keep you awake.
- **Make your bedroom sleep‑friendly**
  - Keep the room cool, dark, and quiet.
  - Use comfortable bedding and a supportive pillow.
- **Watch what you eat and drink**
  - Avoid large meals, caffeine, and alcohol close to bedtime.
- **Stay active during the day**
  - Regular physical activity helps you fall asleep faster, but try not to exercise vigorously right before bed.
- **Manage stress**
  - Try simple relaxation techniques like deep breathing, meditation, or journaling to clear your mind before sleep.
- **Limit naps**
  - If you need to nap, keep it short (20

In [5]:
def classify_intent(user_message):

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": """
                Classify the user's message into exactly one category:
                health_question, habit_tracking, reminder, health_summary.

                Return only the category name.
                """
            },
            {
                "role": "user",
                "content": user_message
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip()


print(classify_intent("I slept for 6 hours yesterday"))

habit_tracking


In [6]:
import json

def extract_health_data(user_message):

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": """
                Extract only information explicitly mentioned by the user.

                Return valid JSON with these fields:
                {
                    "sleep_hours": null,
                    "water_glasses": null,
                    "exercise_minutes": null
                }

                Use null if information is not mentioned.
                Do not guess values.
                """
            },
            {
                "role": "user",
                "content": user_message
            }
        ],
        temperature=0
    )

    return json.loads(response.choices[0].message.content)


data = extract_health_data(
    "I slept for 7 hours and exercised for 30 minutes."
)

print(data)

{'sleep_hours': 7, 'water_glasses': None, 'exercise_minutes': 30}


In [7]:
health_data = {
    "sleep_hours": 0,
    "water_glasses": 0,
    "exercise_minutes": 0
}

In [8]:
def update_health_data(new_data):

    for key, value in new_data.items():

        if value is not None:
            health_data[key] = value

    return health_data


update_health_data({
    "sleep_hours": 8,
    "water_glasses": 6,
    "exercise_minutes": 30
})

print(health_data)

{'sleep_hours': 8, 'water_glasses': 6, 'exercise_minutes': 30}


In [9]:
def log_call(func):

    def wrapper(*args, **kwargs):

        print("Executing:", func.__name__)

        result = func(*args, **kwargs)

        print("Execution completed!")

        return result

    return wrapper


@log_call
def calculate_weekly_sleep(daily_sleep):

    return sum(daily_sleep) / len(daily_sleep)


print(calculate_weekly_sleep([7, 8, 6, 7, 8, 9, 7]))

Executing: calculate_weekly_sleep
Execution completed!
7.428571428571429


In [10]:
from openai import RateLimitError, APIConnectionError, AuthenticationError

class HealthTrackAgent:

    def __init__(self, max_turns=5):

        self.history = []

        self.max_turns = max_turns

    def chat(self, user_message):

        try:

            intent = classify_intent(user_message)

            if intent == "habit_tracking":

                extracted_data = extract_health_data(user_message)

                update_health_data(extracted_data)

            self.history.append({
                "role": "user",
                "content": user_message
            })

            messages = [
                {"role": "system", "content": SYSTEM_PROMPT}
            ]

            messages.extend(self.history[-self.max_turns * 2:])

            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                temperature=0.3
            )

            answer = response.choices[0].message.content

            self.history.append({
                "role": "assistant",
                "content": answer
            })

            return answer

        except RateLimitError:
            return "API rate limit reached. Please try again later."

        except APIConnectionError:
            return "Connection problem. Please check your internet."

        except AuthenticationError:
            return "API key authentication failed."

        except Exception as e:
            return f"An error occurred: {e}"


agent = HealthTrackAgent()

print(agent.chat("I slept for 7 hours and exercised for 30 minutes."))

Great job on your day! Here’s a quick look at what you did:

- **Sleep:** 7 hours – right in the healthy range (7‑9 hours for most adults).  
- **Exercise:** 30 minutes – a solid start; aim for at least 150 minutes of moderate activity each week.

**Tips to keep the momentum:**
- Keep a regular bedtime and wake‑time to support consistent sleep quality.  
- Mix in different activities (walking, cycling, strength work) to stay motivated.  
- Pair your activity with balanced meals—protein, whole grains, fruits, and veggies.

Let me know if you’d like ideas for workouts, sleep‑hygiene tricks, or nutrition tips!


In [12]:
print(agent.chat("My name is Sri Venkatesh."))

print(agent.chat("I am studying Artificial Intelligence."))

print(agent.chat("What is my name and what am I studying?"))

Got it—your name is **Sri Venkatesh**. How can I help you with your health or wellness today?
Thanks for confirming, **Sri Venkatesh**! You’re studying **Artificial Intelligence**. Let me know if you’d like any health‑oriented tips to support your studies.
Your name is **Sri Venkatesh**, and you are studying **Artificial Intelligence**.


In [13]:
import ipywidgets as widgets
from IPython.display import display, clear_output

chat_input = widgets.Text(
    placeholder="Ask HealthTrack AI...",
    description="You:"
)

send_button = widgets.Button(
    description="Send",
    button_style="success"
)

output = widgets.Output()


def send_message(button):

    message = chat_input.value.strip()

    if not message:
        return

    with output:

        print("You:", message)

        response = agent.chat(message)

        print("HealthTrack AI:", response)
        print("-" * 50)

    chat_input.value = ""


send_button.on_click(send_message)

display(chat_input, send_button, output)

Text(value='', description='You:', placeholder='Ask HealthTrack AI...')

Button(button_style='success', description='Send', style=ButtonStyle())

Output()